<a href="https://colab.research.google.com/github/Aditya-Bang/MinesweeperGPT/blob/main/src/finetuning/MinesweeperGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!nvcc --version
!uname -a

Wed Aug 27 14:33:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%capture
!pip install -qqq torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
%%capture
!pip install -qqq unsloth==2025.6.2 unsloth_zoo==2025.6.1 trl==0.18.1 vllm==0.8.5.post1 xformers==0.0.29.post2 triton==3.2.0 accelerate==1.7.0 transformers==4.51.3 torchao==0.12.0 wandb

In [ ]:
!pip list | grep -E 'torch|triton|vllm|unsloth|transformers|xformers|accelerate|trl|wandb'

accelerate                               1.7.0
fastrlock                                0.8.3
sentence-transformers                    5.1.0
torch                                    2.6.0+cu124
torchao                                  0.12.0
torchaudio                               2.6.0+cu124
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.21.0+cu124
transformers                             4.51.3
triton                                   3.2.0
trl                                      0.18.1
unsloth                                  2025.6.2
unsloth_zoo                              2025.6.1
vllm                                     0.8.5.post1
wandb                                    0.21.1
xformers                                 0.0.29.post2


In [ ]:
%%capture
!unzip train-data.zip -d train-data
!unzip test-data.zip -d test-data

In [ ]:
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 08-27 14:39:28 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 08-27 14:39:28 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
max_seq_length = 512  # Can increase for longer reasoning traces
lora_rank = 32         # Larger rank = smarter, but slower

# Load model + tokenizer with vLLM acceleration
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,       # False for LoRA 16bit
    fast_inference = True,      # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.45, # Reduce if out of memory
)

==((====))==  Unsloth 2025.6.2: Fast Qwen3 patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen3-4b-unsloth-bnb-4bit with actual GPU utilization = 44.57%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 3.82 GB. Also swap space = 0 GB.
WARNING 08-27 14:39:54 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 08-27 14:40:21 [config.py:717] This model supports multiple tasks: {'generate', 'classify', 'score', 'reward', '

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
class NoThinkTokenizerWrapper:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __getattr__(self, name):
        return getattr(self.tokenizer, name)

    def __call__(self, *args, **kwargs):
        return self.tokenizer(*args, **kwargs)

    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True, **kwargs):
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,   # force disable thinking
            **kwargs
        )

no_think_tokenizer = NoThinkTokenizerWrapper(tokenizer)

In [ ]:
# src/models.py
from dataclasses import dataclass
from typing import List

@dataclass
class MinesweeperExample:
    input: str
    board_state: List[List[str]]
    hidden_state: List[List[str]]

@dataclass
class Move:
    row: int
    col: int
    action: str

In [ ]:
# src/utils.py
from pathlib import Path

def get_base_directory() -> Path:
    """
    Returns the base directory path of the project.
    """
    return Path.cwd()


In [ ]:
# src/globals.py
TRAINING_ROWS = 5
TRAINING_COLS = 5
TRAINING_MIN_MINES = 5
TRAINING_MAX_MINES = 8

In [ ]:
SYSTEM_PROMPT = f"""
You are a Minesweeper assistant.
The game board is always {TRAINING_ROWS}x{TRAINING_COLS} in size.
You will be given ONLY the current board state as input from the user.

Your task: Suggest exactly ONE valid next move for the minesweeper board given by the user.

Move format rules (must follow exactly one of these two):
1. "row: NUM, col: NUM, action: reveal"       → to reveal a cell
2. "row: NUM, col: NUM, action: flag"         → to flag a cell as a mine

Board representation:
- '*' means the tile has not been revealed yet.
- Numbers 0–8 show how many mines are adjacent to that square.
- 'F' means the tile has already been flagged as a mine.
- The board will be displayed as a grid of symbols only.

Important condition:
- You may only suggest to reveal of flag a tile that is not already revealed, i.e. contains '*'.
- Do NOT suggest moves on numbers or flagged tiles, as these have already been revealed or correctly flagged.

Summary:
- Suggest one valid move next with the format "row: NUM, col: NUM, action: reveal" or "row: NUM, col: NUM, action: flag", where NUM is an integer in the range [1, {TRAINING_ROWS}] for rows and [1, {TRAINING_COLS}] for columns.
- Only output the valid move.
"""

def add_row_numbers(board_str: str) -> str:
    lines = board_str.splitlines()
    numbered_lines = [f"Row {i}: {line}" for i, line in enumerate(lines, start=1) if line.strip()]
    return "\n".join(numbered_lines)

def format_example(board: str) -> dict:
    formatted_board: str = add_row_numbers(board)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": formatted_board + "\n/no_think"},
    ]


In [ ]:
messages = format_example("""
0 0 2 F *
0 0 3 * *
0 0 2 * *
1 1 1 * *
* * * * *
""")

# Apply chat template, enabling thinking mode
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Tokenize input
inputs = tokenizer([text], return_tensors="pt").to(model.device)
print("Number of tokens:", len(inputs["input_ids"][0]))


# Generate
generated_ids = model.generate(
    **inputs,
    max_new_tokens=512,
)

# Extract only the new tokens
output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist()
decoded_output = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
print("output:", decoded_output)


Number of tokens: 365
output: <think>

</think>

row: 5, col: 4, action: reveal


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Remove QKVO if out of memory
    lora_alpha=lora_rank*2,
    use_gradient_checkpointing="unsloth",  # Enable long context finetuning
    random_state=3407,
)

Unsloth 2025.6.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
# src/finetuning/dataset.py
from pathlib import Path
from typing import List, Dict, Any
import random
from datasets import Dataset


class MinesweeperDatasetLoader:
    def __init__(self, data_dir: str = "data"):
        self.base_dir: Path = get_base_directory()
        self.data_dir: Path = (self.base_dir / data_dir).resolve()
        print(f"Using data directory: {self.data_dir}")
        if not self.data_dir.exists():
            print(f"Data directory does not exist: {self.data_dir}")
        else:
            print(f"Data directory exists: {self.data_dir}")
        self.games = sorted([g for g in self.data_dir.glob("game*") if g.is_dir()])
        if not self.games:
            print(f"No game directories found in {self.data_dir}")
        else:
            print(f"Found {len(self.games)} game directories in {self.data_dir}")

    def load_examples(self) -> List[MinesweeperExample]:
        examples: List[MinesweeperExample] = []
        for game in self.games:
            step_files = sorted(game.glob("step*.txt"), key=lambda p: int(p.stem[4:]))
            hidden_state: List[List[str]] = [
                line.split() for line in (game / "hidden_state.txt").read_text().splitlines() if line.strip()
            ]

            for i in range(len(step_files) - 1):
                state_str = step_files[i].read_text()
                board_state: List[List[str]] = [
                    line.split() for line in state_str.splitlines() if line.strip()
                ]
                # next_state = step_files[i+1].read_text()

                # Find action: difference between states (TODO: store all possible valid actions)
                # action = self._extract_action(state, next_state)
                examples.append(MinesweeperExample(input=state_str, board_state=board_state, hidden_state=hidden_state))
        random.shuffle(examples)
        return examples

    # def _extract_action(self, prev_state: str, next_state: str) -> str:
    #     prev_lines = prev_state.splitlines()
    #     next_lines = next_state.splitlines()
    #     for r, (pl, nl) in enumerate(zip(prev_lines, next_lines)):
    #         for c, (pc, nc) in enumerate(zip(pl, nl)):
    #             if pc != nc:
    #                 return f"{r} {c}" if nc != "F" else f"{r} {c} f"
    #     raise ValueError("No difference found between states; every state transition must have a difference.")

    def to_hf_dataset(self) -> Dataset:
        """
        Convert Minesweeper examples to a Hugging Face Dataset object
        suitable for GRPO training.
        """
        examples: List[MinesweeperExample] = self.load_examples()

        hf_data: List[Dict[str, Any]] = [
            {
                "prompt": format_example(ex.input),
                "board_state": ex.board_state,
                "hidden_state": ex.hidden_state,
            }
            for ex in examples
        ]

        return Dataset.from_list(hf_data)


In [ ]:
from datasets import Dataset
from pprint import pprint

train_dataset_loader = MinesweeperDatasetLoader(data_dir="train-data")
train_dataset: Dataset = train_dataset_loader.to_hf_dataset()
for i in range(2):
    pprint(train_dataset[i])

test_dataset_loader = MinesweeperDatasetLoader(data_dir="test-data")
test_dataset: Dataset = test_dataset_loader.to_hf_dataset()
for i in range(2):
    pprint(test_dataset[i])

Using data directory: /content/train-data
Data directory exists: /content/train-data
Found 90 game directories in /content/train-data
{'board_state': [['1', '1', '1', '0', '0'],
                 ['*', 'F', '2', '0', '0'],
                 ['*', 'F', '5', '2', '1'],
                 ['*', 'F', 'F', 'F', '1'],
                 ['*', '*', 'F', '3', '1']],
 'hidden_state': [['1', '1', '1', '0', '0'],
                  ['2', 'M', '2', '0', '0'],
                  ['3', 'M', '5', '2', '1'],
                  ['2', 'M', 'M', 'M', '1'],
                  ['1', '3', 'M', '3', '1']],
 'prompt': [{'content': '\n'
                        'You are a Minesweeper assistant.\n'
                        'The game board is always 5x5 in size.\n'
                        'You will be given ONLY the current board state as '
                        'input from the user.\n'
                        '\n'
                        'Your task: Suggest exactly ONE valid next move for '
                        'the m

In [ ]:
from tqdm import tqdm

def count_tokens(example):
    # Turn messages into text using your chat template
    text = tokenizer.apply_chat_template(
        example["prompt"],  # or however your dataset stores conversations
        tokenize=False,
        add_generation_prompt=True,
    )
    # Tokenize and return number of tokens
    return len(tokenizer(text)["input_ids"])

# Collect lengths for the whole dataset
lengths = [count_tokens(example) for example in tqdm(train_dataset)]

print("Max prompt length:", max(lengths))
print("Avg prompt length:", sum(lengths) / len(lengths))
print("Some samples:", lengths[:10])


100%|██████████| 963/963 [00:01<00:00, 569.80it/s]

Max prompt length: 378
Avg prompt length: 370.32294911734164
Some samples: [377, 374, 363, 367, 360, 367, 370, 372, 373, 375]


In [ ]:
# src/finetuning/rewards.py
import re
from typing import List, Optional, Tuple, Dict

global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5


def is_output_valid(move_str: str) -> Optional[Tuple[int, int, str]]:
    """
    Check if the output move string matches the expected format,
    even if arbitrary characters precede it.
    """
    pattern = r"row:\s*(\d+),\s*col:\s*(\d+),\s*action:\s*(reveal|flag)"
    return re.search(pattern, move_str.strip(), re.DOTALL)


def parse_move(move_str: str) -> Tuple[Optional[int], Optional[int], Optional[str]]:
    """
    Parse a move string of the format: "row: NUM, col: NUM, action: reveal/flag"
    Returns a tuple: (row, col, action)
    """
    match = is_output_valid(move_str)
    if match:
        row, col, action = match.groups()
        return int(row) - 1, int(col) - 1, action  # Convert to 0-indexed
    return None, None, None


def move_square_in_bounds(row: int, col: int) -> bool: # row, col 0-indexed
    """
    Check if the given row and column are within the board bounds.
    """
    return 0 <= row < TRAINING_ROWS and 0 <= col < TRAINING_COLS



def reward_format_correct(
        completions: List[List[Dict[str, str]]],
        **kwargs
    ) -> List[float]:
    """
    Reward 1: Check if the move follows the correct format.
    """
    responses = [completion[0]["content"] for completion in completions]
    return [1.0 if is_output_valid(response) else 0.0 for response in responses]


def reward_valid_cell(
        completions: List[List[Dict[str, str]]],
        board_state: List[List[List[str]]],
        **kwargs
    ) -> List[float]:
    """
    Reward 2: Reward if the move targets an unrevealed cell ('*').
    board: List of strings representing the current board state.
    """
    scores = []
    responses = [completion[0]["content"] for completion in completions]
    for response, b_state in zip(responses, board_state):
        row, col, action = parse_move(response)
        if row is None:
            scores.append(0.0)
            continue
        if not move_square_in_bounds(row, col):
            scores.append(0.0)
            continue
        if b_state[row][col] == '*':
            scores.append(2.5)
            continue
        scores.append(0.0)
    return scores

def reward_logical_move(
        prompts: List[List[Dict[str, str]]],
        completions: List[List[Dict[str, str]]],
        board_state: List[List[List[str]]],
        hidden_state: List[List[List[str]]],
        **kwargs
    ) -> List[float]:
    """
    Reward 3: Give score if the move is logical.
    A move is rewarded (score=3) if the chosen square is unrevealed ('*')
    AND at least one of the 8 surrounding squares is revealed (not '*').
    """
    scores = []
    responses = [completion[0]["content"] for completion in completions]

    # directions for 8 neighbors
    neighbors = [(-1, -1), (-1, 0), (-1, 1),
                 (0, -1),           (0, 1),
                 (1, -1),  (1, 0),  (1, 1)]

    for response, b_state, h_state in zip(responses, board_state, hidden_state):
        row, col, action = parse_move(response)
        if row is None or not move_square_in_bounds(row, col):
            scores.append(0.0)
            continue

        # must be unrevealed
        if b_state[row][col] != '*':
            scores.append(0.0)
            continue

        # check surrounding squares
        logical = 0
        for dr, dc in neighbors:
            nr, nc = row + dr, col + dc
            if move_square_in_bounds(nr, nc):
                if b_state[nr][nc] != '*':
                    logical += 1

        scores.append(min(float(logical), 3.0))

    return scores

def reward_correct_move(
        prompts: List[List[Dict[str, str]]],
        completions: List[List[Dict[str, str]]],
        board_state: List[List[List[str]]],
        hidden_state: List[List[List[str]]],
        **kwargs
    ) -> List[float]:
    """
    Reward 4: Reward if the move reveals an empty square or flags a mine correctly.
    Additional rule: If the move is not logical (none of the surrounding squares
    is a number or 'F'), then reward = 0.

    hidden_state: List of strings representing the ground truth board with mines.
    """
    scores = []
    responses = [completion[0]["content"] for completion in completions]

    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f" Reward 4 Debugging Info (every {PRINT_EVERY_STEPS} steps) " + '*'*20 + '\n' +
            f"Prompt:\n{prompts[0][-1]['content']}\n" +
            f"Responses:\n{responses[0]}\n"
        )
    PRINTED_TIMES += 1

    # directions for 8 neighbors
    neighbors = [(-1, -1), (-1, 0), (-1, 1),
                 (0, -1),           (0, 1),
                 (1, -1),  (1, 0),  (1, 1)]

    for response, b_state, h_state in zip(responses, board_state, hidden_state):
        row, col, action = parse_move(response)
        if row is None or not move_square_in_bounds(row, col):
            scores.append(0.0)
            continue
        if b_state[row][col] != '*':
            scores.append(0.0)
            continue

        # ✅ Check if move is "logical"
        logical = False
        for dr, dc in neighbors:
            nr, nc = row + dr, col + dc
            if move_square_in_bounds(nr, nc):
                neighbor_val = b_state[nr][nc]
                if neighbor_val.isdigit() or neighbor_val == "F":
                    logical = True
                    break

        if not logical:
            scores.append(0.0)
            continue

        # ✅ Only reward logical AND correct moves
        if action == "reveal":
            scores.append(3.0 if h_state[row][col] != 'M' else 0.0)
        elif action == "flag":
            scores.append(3.0 if h_state[row][col] == 'M' else 0.0)
        else:
            scores.append(0.0)

    return scores

In [ ]:
example = train_dataset[0]

b_state = example["board_state"]
h_state = example["hidden_state"]

pprint(b_state)
pprint(h_state)

completions = [
    [{"content": "<think><\think> row: 1, col: 1, action: reveal```"}],
    [{"content": "row: 1, col: 1, action: flag"}],
    [{"content": "row: 5, col: 5, action: reveal"}],
    [{"content": "asdfasdfrow: 4, col: 5, action: flag"}],
]

print("Reward 1 (format):", reward_format_correct(completions))
print("Reward 2 (valid cell):", reward_valid_cell(completions, [b_state]*4))
print("Reward 3 (logical move):", reward_logical_move(
    prompts=completions,
    completions=completions,
    board_state=[b_state]*4,
    hidden_state=[h_state]*4
))
print("Reward 4 (correct move):", reward_correct_move(
    prompts=completions,
    completions=completions,
    board_state=[b_state]*4,
    hidden_state=[h_state]*4
))

[['0', '0', '0', '0', '0'],
 ['1', '1', '1', '0', '0'],
 ['*', 'F', '1', '1', '1'],
 ['*', '4', '2', '1', 'F'],
 ['*', '*', '1', '1', '1']]
[['0', '0', '0', '0', '0'],
 ['1', '1', '1', '0', '0'],
 ['2', 'M', '1', '1', '1'],
 ['M', '4', '2', '1', 'M'],
 ['M', 'M', '1', '1', '1']]
Reward 1 (format): [1.0, 1.0, 1.0, 1.0]
Reward 2 (valid cell): [0.0, 0.0, 0.0, 0.0]
Reward 3 (logical move): [0.0, 0.0, 0.0, 0.0]
******************** Reward 4 Debugging Info (every 5 steps) ********************
Prompt:
<think><	hink> row: 1, col: 1, action: reveal```
Responses:
<think><	hink> row: 1, col: 1, action: reveal```

Reward 4 (correct move): [0.0, 0.0, 0.0, 0.0]


In [ ]:
print(is_output_valid("<think></think> row: 1, col: 1, action: reveal'''"))

<re.Match object; span=(16, 46), match='row: 1, col: 1, action: reveal'>


In [ ]:
from typing import List, Dict, Optional
import re
from pprint import pprint
from datasets import Dataset
from vllm import SamplingParams

max_prompt_length=420

class LLMTester:
    def __init__(self, model, tokenizer, dataset: Dataset, lora_request = None):
        self.model = model
        self.tokenizer = tokenizer
        self.dataset: Dataset = dataset
        self.lora_request = lora_request

    def test_llm(self, verbose: bool = False):
        moves_correct = 0
        for example in self.dataset:
            prompt = example["prompt"]
            board_state: List[List[str]] = example["board_state"]
            hidden_state: List[List[str]] = example["hidden_state"]

            llm_move: str = self.generate_llm_move(prompt)
            if verbose:
                print(f"Board state:")
                pprint(board_state)
                print(f"Hidden state:")
                pprint(hidden_state)
                print(f"LLM Move:\n{llm_move}")

            parsed_llm_move: Optional[Move] = self.parse_llm_move(llm_move)
            if not parsed_llm_move:
                if verbose: print(f"Error parsing llm move: {llm_move}")
                continue

            if not self.validate_llm_move(parsed_llm_move, board_state):
                if verbose: print(f"Invalid move: {parsed_llm_move}")
                continue

            if not self.verify_llm_move(parsed_llm_move, hidden_state):
                if verbose: print(f"Move incorrect: {parsed_llm_move}")
                continue

            moves_correct += 1
            if verbose: print(f"Move correct!")

        print(f"Moves correct: {moves_correct}/{len(self.dataset)}")
        print(f"Moves incorrect: {len(self.dataset) - moves_correct}/{len(self.dataset)}")

    def generate_llm_move(self, prompt: List[Dict[str, str]]) -> str:
        text = self.tokenizer.apply_chat_template(
            prompt,
            tokenize=False,
            add_generation_prompt=True,
        )
        sampling_params = SamplingParams(
            temperature=0.0,
            top_k=-1,
            top_p=1.0,
            max_tokens=max_seq_length-max_prompt_length,
        )
        output: str = model.fast_generate(
            text,
            sampling_params=sampling_params,
            lora_request=self.lora_request,
        )[0].outputs[0].text
        return output
        # input_tokens = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        # generated_ids = self.model.generate(
        #     **input_tokens,
        #     max_new_tokens=512,
        # )
        # output_ids = generated_ids[0][len(input_tokens.input_ids[0]):].tolist()
        # output: str = self.tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
        # return output

    def parse_llm_move(self, llm_move: str) -> Optional[Move]:
        pattern = r"row:\s*(\d+),\s*col:\s*(\d+),\s*action:\s*(reveal|flag)"
        match = re.search(pattern, llm_move.strip(), re.DOTALL)
        if match:
            row, col, action = match.groups()
            return Move(row=int(row)-1, col=int(col)-1, action=action)  # 0-indexed
        return None

    def validate_llm_move(self, move: Move, board_state: List[List[str]]) -> bool:
        rows = len(board_state)
        cols = len(board_state[0])
        if not (0 <= move.row < rows and 0 <= move.col < cols):
            return False  # Out of bounds
        if board_state[move.row][move.col] != "*":  # Already revealed or flagged
            return False
        return True

    def verify_llm_move(self, move: Move, hidden_state: List[List[str]]) -> bool:
        cell_value = hidden_state[move.row][move.col]
        if move.action == "reveal":
            return cell_value != "M"  # Should not reveal a mine
        elif move.action == "flag":
            return cell_value == "M"  # Should only flag mines
        return False


# Example usage
llm_tester = LLMTester(model=model, tokenizer=tokenizer, dataset=test_dataset.select(range(20)))
llm_tester.test_llm(verbose=True)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['F', 'F', '2', '*', '*'],
 ['2', '4', 'F', '*', '*'],
 ['0', '2', 'F', '*', '*'],
 ['0', '1', '2', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '2', '1', '0'],
 ['2', '4', 'M', '2', '0'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '2', '2', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', '3', '2', '*'],
 ['2', '2', '1', '*', '*'],
 ['0', '0', '1', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '3', 'M', 'M'],
 ['M', 'M', '3', '2', '2'],
 ['2', '2', '1', '0', '0'],
 ['0', '0', '1', '1', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move incorrect: Move(row=0, col=3, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Board state:
[['F', '2', '0', '0', '0'],
 ['F', '3', '1', '1', '0'],
 ['*', '*', 'F', '3', '2'],
 ['*', '*', '*', 'F', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['M', '2', '0', '0', '0'],
 ['M', '3', '1', '1', '0'],
 ['1', '2', 'M', '3', '2'],
 ['0', '2', '3', 'M', 'M'],
 ['0', '1', 'M', '3', '2']]
LLM Move:
<think>

</think>

row: 3, col: 1, action: flag
Move incorrect: Move(row=2, col=0, action='flag')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '1', '0', '0', '0'],
 ['F', '2', '0', '1', '1'],
 ['F', '3', '0', '2', 'F'],
 ['F', '2', '0', '3', 'F'],
 ['1', '1', '0', '2', 'F']]
Hidden state:
[['1', '1', '0', '0', '0'],
 ['M', '2', '0', '1', '1'],
 ['M', '3', '0', '2', 'M'],
 ['M', '2', '0', '3', 'M'],
 ['1', '1', '0', '2', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['F', 'F', '1', '1', '*'],
 ['2', '2', '1', '1', '*'],
 ['0', '0', '1', '2', '2'],
 ['2', '3', '3', 'F', '1'],
 ['F', 'F', 'F', '2', '1']]
Hidden state:
[['M', 'M', '1', '1', '1'],
 ['2', '2', '1', '1', 'M'],
 ['0', '0', '1', '2', '2'],
 ['2', '3', '3', 'M', '1'],
 ['M', 'M', 'M', '2', '1']]
LLM Move:
<think>

</think>

row: 1, col: 5, action: reveal
Move correct!
Moves correct: 10/20
Moves incorrect: 10/20


In [ ]:
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adityabang15 (adityabang15-university-of-waterloo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
def stop_here():
    raise KeyboardInterrupt

In [ ]:
from trl import GRPOConfig, GRPOTrainer
max_prompt_length = 420

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
    max_tokens = max_seq_length - max_prompt_length
)

training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,

    learning_rate = 3.5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    # learning_rate=1e-6,
    # weight_decay=0.01,
    # warmup_ratio=0.1,
    # lr_scheduler_type="cosine_with_restarts",

    optim="paged_adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Increase to 4 for smoother training
    num_generations=3,  # Decrease if out of memory
    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,
    num_train_epochs = 1, # Set to 1 for a full training run
    # max_steps=100,
    save_steps=10,
    max_grad_norm=0.1,
    # report_to="none",  # Can use Weights & Biases
    report_to=["wandb"],
    output_dir="outputs",

    # fp16_full_eval = True,
    # per_device_eval_batch_size=3,
    # eval_accumulation_steps=1,
    # eval_strategy="steps",
    # eval_steps=100,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_format_correct,
        reward_valid_cell,
        reward_correct_move
    ],
    args=training_args,
    train_dataset=train_dataset,
    # eval_dataset = test_dataset.select(range(27)),
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 3


In [ ]:
# initial_state_dict = {
#     name: param.cpu().clone()
#     for name, param in trainer.model.named_parameters()
# }

# print("Starting training...")
# trainer.evaluate()
wandb.init(project="MinesweeperGPT")
trainer.train()
# print("Training finished.")

# final_state_dict = {
#     name: param.cpu()
#     for name, param in trainer.model.named_parameters()
# }

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 963 | Num Epochs = 1 | Total steps = 240
O^O/ \_/ \    Batch size per device = 3 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (3 x 4 x 1) = 12
 "-____-"     Trainable parameters = 66,060,288/4,000,000,000 (1.65% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_format_correct / mean,rewards / reward_format_correct / std,rewards / reward_valid_cell / mean,rewards / reward_valid_cell / std,rewards / reward_correct_move / mean,rewards / reward_correct_move / std
1,0.000000,5.000000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000000,1.000000,0.000000,2.500000,0.000000,1.500000,1.566699
2,0.000000,4.250000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000000,1.000000,0.000000,2.500000,0.000000,0.750000,1.356801
3,0.000000,5.500000,0.433013,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000000,1.000000,0.000000,2.500000,0.000000,2.000000,1.477098
4,0.000000,4.041667,0.360844,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000024,1.000000,0.000000,2.291667,0.721688,0.750000,1.356801
5,0.000000,3.500000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000027,1.000000,0.000000,2.500000,0.000000,0.000000,0.000000
6,0.000000,5.000000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000002,1.000000,0.000000,2.500000,0.000000,1.500000,1.566699
7,0.000000,4.250000,0.866025,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000172,1.000000,0.000000,2.500000,0.000000,0.750000,1.356801
8,0.000000,5.750000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000159,1.000000,0.000000,2.500000,0.000000,2.250000,1.356801
9,0.000000,5.000000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000010,1.000000,0.000000,2.500000,0.000000,1.500000,1.566699
10,0.000000,5.000000,0.000000,18.000000,18.000000,18.000000,0.000000,18.000000,18.000000,18.000000,0.000093,1.000000,0.000000,2.500000,0.000000,1.500000,1.566699


******************** Reward 4 Debugging Info (every 5 steps) ********************
Prompt:
Row 1: 0 1 * * *
Row 2: 0 1 * * *
Row 3: 2 2 * * *
Row 4: F F * * *
Row 5: * * * * *
/no_think
Responses:
<think>

</think>

row: 1, col: 3, action: reveal

******************** Reward 4 Debugging Info (every 5 steps) ********************
Prompt:
Row 1: * * 1 0 0
Row 2: * F 2 0 0
Row 3: * F 5 2 1
Row 4: * F F F 1
Row 5: * * * * *
/no_think
Responses:
<think>

</think>

row: 1, col: 1, action: reveal

******************** Reward 4 Debugging Info (every 5 steps) ********************
Prompt:
Row 1: 0 0 2 F *
Row 2: 1 1 2 F *
Row 3: F 4 3 2 *
Row 4: F * * * *
Row 5: * * * * *
/no_think
Responses:
<think>

</think>

row: 1, col: 5, action: reveal

******************** Reward 4 Debugging Info (every 5 steps) ********************
Prompt:
Row 1: 0 1 * * *
Row 2: 0 1 F 3 *
Row 3: 0 1 1 3 2
Row 4: 2 2 1 2 F
Row 5: F F 1 2 F
/no_think
Responses:
<think>

</think>

row: 1, col: 3, action: reveal

************

TrainOutput(global_step=240, training_loss=0.0003790361602189485, metrics={'train_runtime': 3778.354, 'train_samples_per_second': 0.255, 'train_steps_per_second': 0.064, 'total_flos': 0.0, 'train_loss': 0.0003790361602189485})

In [ ]:
model.save_lora("grpo_saved_lora_9")

In [ ]:
stop_here()

KeyboardInterrupt: 

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora_9/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [ ]:
from vllm import SamplingParams

text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": """
Row 1: F F F 1 0
Row 2: 2 4 3 3 1
Row 3: 0 1 F 4 F
Row 4: 0 1 2 F F
Row 5: 0 0 1 * *
/no_think
"""},
    ],
    tokenize=False,
    add_generation_prompt=True,
    # enable_thinking=False,
)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=0.95,
    max_tokens=512,
)

# BEFORE training
output_before = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=None
    )[0]
    .outputs[0]
    .text
)

print("Output before training:")
print(output_before)

# AFTER training (or after loading LoRA)
output_after = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("grpo_saved_lora_9"),
    )[0]
    .outputs[0]
    .text
)

print("\nOutput after training / LoRA:")
print(output_after)


In [ ]:
import torch

def compare_models(model1, model2):
    diffs = {}
    for (name1, param1), (name2, param2) in zip(model1.named_parameters(), model2.named_parameters()):
        if "weight" in name1 or "bias" in name1:  # only compare learned params
            if param1.dtype in [torch.float16, torch.float32, torch.float64]:
                diff = torch.norm(param1.data.float() - param2.data.float()).item()
                diffs[name1] = diff
    return diffs

# Move both models to CPU for comparison
model_cpu = model.to("cpu")
trained_model_cpu = trained_model.to("cpu")

diffs = compare_models(model_cpu, trained_model_cpu)

# Print a few sample diffs
for layer, d in list(diffs.items())[:10]:
    print(f"{layer}: {d}")

# Move back to GPU when done
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
trained_model = trained_model.to(device)


In [ ]:
llm_tester = LLMTester(model, tokenizer, test_dataset, lora_request=model.load_lora("grpo_saved_lora_9"))
llm_tester.test_llm(verbose=True)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', 'F', '*', '*'],
 ['2', '3', '2', '3', '*'],
 ['0', '0', '0', '3', '*'],
 ['0', '0', '0', '2', '*']]
Hidden state:
[['2', '3', '3', 'M', 'M'],
 ['M', 'M', 'M', '4', '3'],
 ['2', '3', '2', '3', 'M'],
 ['0', '0', '0', '3', 'M'],
 ['0', '0', '0', '2', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move incorrect: Move(row=0, col=3, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['0', '1', '*', '*', '*'],
 ['0', '2', 'F', '*', '*'],
 ['0', '3', 'F', '*', '*'],
 ['0', '2', 'F', '2', '*'],
 ['0', '1', '1', '1', '*']]
Hidden state:
[['0', '1', '2', '4', 'M'],
 ['0', '2', 'M', 'M', 'M'],
 ['0', '3', 'M', '5', '2'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '1', '1', '0']]
LLM Move:
<think>

</think>

row: 1, col: 3, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Board state:
[['0', '1', '2', '4', '*'],
 ['0', '2', 'F', 'F', 'F'],
 ['0', '3', 'F', '5', '2'],
 ['0', '2', 'F', '2', '0'],
 ['0', '1', '1', '1', '0']]
Hidden state:
[['0', '1', '2', '4', 'M'],
 ['0', '2', 'M', 'M', 'M'],
 ['0', '3', 'M', '5', '2'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '1', '1', '0']]
LLM Move:
<think>

</think>

row: 1, col: 5, action: reveal
Move incorrect: Move(row=0, col=4, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '2', '1', '0'],
 ['*', '3', 'F', '1', '0'],
 ['1', '2', '1', '2', '1'],
 ['0', '0', '0', '2', 'F'],
 ['0', '0', '0', '2', 'F']]
Hidden state:
[['2', 'M', '2', '1', '0'],
 ['M', '3', 'M', '1', '0'],
 ['1', '2', '1', '2', '1'],
 ['0', '0', '0', '2', 'M'],
 ['0', '0', '0', '2', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['0', '1', 'F', '3', 'F'],
 ['0', '2', '2', '4', '*'],
 ['0', '1', 'F', '2', '*'],
 ['0', '1', '2', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['0', '1', 'M', '3', 'M'],
 ['0', '2', '2', '4', 'M'],
 ['0', '1', 'M', '2', '1'],
 ['0', '1', '2', '3', '2'],
 ['0', '0', '1', 'M', 'M']]
LLM Move:
<think>

</think>

row: 5, col: 5, action: reveal
Move incorrect: Move(row=4, col=4, action='reveal')
Moves correct: 96/175
Moves incorrect: 79/175


In [ ]:
llm_tester = LLMTester(model, tokenizer, test_dataset)
llm_tester.test_llm(verbose=True)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['F', 'F', '2', '*', '*'],
 ['2', '4', 'F', '*', '*'],
 ['0', '2', 'F', '*', '*'],
 ['0', '1', '2', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '2', '1', '0'],
 ['2', '4', 'M', '2', '0'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '2', '2', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', '3', '2', '*'],
 ['2', '2', '1', '*', '*'],
 ['0', '0', '1', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '3', 'M', 'M'],
 ['M', 'M', '3', '2', '2'],
 ['2', '2', '1', '0', '0'],
 ['0', '0', '1', '1', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move incorrect: Move(row=0, col=3, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Board state:
[['*', 'F', '2', '0', '0'],
 ['*', 'F', '2', '0', '0'],
 ['*', '*', '2', '1', '1'],
 ['*', '*', '*', '*', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['2', 'M', '2', '0', '0'],
 ['2', 'M', '2', '0', '0'],
 ['1', '1', '2', '1', '1'],
 ['2', '2', '3', 'M', '3'],
 ['M', 'M', '3', 'M', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 5, action: reveal
Invalid move: Move(row=0, col=4, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '1', '1', '0', '0'],
 ['*', 'F', '1', '1', '1'],
 ['*', '3', '2', '1', 'F'],
 ['*', 'F', '2', '1', '1'],
 ['*', 'F', '2', '0', '0']]
Hidden state:
[['1', '1', '1', '0', '0'],
 ['1', 'M', '1', '1', '1'],
 ['3', '3', '2', '1', 'M'],
 ['M', 'M', '2', '1', '1'],
 ['M', 'M', '2', '0', '0']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', 'F', '2', '0', '0'],
 ['2', 'F', '2', '0', '0'],
 ['1', '1', '2', '1', '1'],
 ['2', '2', '3', 'F', '3'],
 ['F', 'F', '*', '*', '*']]
Hidden state:
[['2', 'M', '2', '0', '0'],
 ['2', 'M', '2', '0', '0'],
 ['1', '1', '2', '1', '1'],
 ['2', '2', '3', 'M', '3'],
 ['M', 'M', '3', 'M', 'M']]
LLM Move:
<think>

</think>

row: 5, col: 3, action: reveal
Move correct!
Moves correct: 96/175
Moves incorrect: 79/175


In [ ]:
wandb.finish()

In [ ]:
llm_tester = LLMTester(model, tokenizer, test_dataset, lora_request=model.load_lora("grpo_saved_lora_8"))
llm_tester.test_llm(verbose=True)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['F', 'F', '2', '*', '*'],
 ['2', '4', 'F', '*', '*'],
 ['0', '2', 'F', '*', '*'],
 ['0', '1', '2', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '2', '1', '0'],
 ['2', '4', 'M', '2', '0'],
 ['0', '2', 'M', '2', '0'],
 ['0', '1', '2', '2', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 4, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '*', '*', '*', '*'],
 ['F', 'F', '3', '2', '*'],
 ['2', '2', '1', '*', '*'],
 ['0', '0', '1', '*', '*'],
 ['0', '0', '1', '*', '*']]
Hidden state:
[['M', 'M', '3', 'M', 'M'],
 ['M', 'M', '3', '2', '2'],
 ['2', '2', '1', '0', '0'],
 ['0', '0', '1', '1', '1'],
 ['0', '0', '1', 'M', '1']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move incorrect: Move(row=0, col=0, action='reveal')


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Board state:
[['*', 'F', '2', '0', '0'],
 ['*', 'F', '2', '0', '0'],
 ['*', '*', '2', '1', '1'],
 ['*', '*', '*', '*', '*'],
 ['*', '*', '*', '*', '*']]
Hidden state:
[['2', 'M', '2', '0', '0'],
 ['2', 'M', '2', '0', '0'],
 ['1', '1', '2', '1', '1'],
 ['2', '2', '3', 'M', '3'],
 ['M', 'M', '3', 'M', 'M']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', '1', '1', '0', '0'],
 ['*', 'F', '1', '1', '1'],
 ['*', '3', '2', '1', 'F'],
 ['*', 'F', '2', '1', '1'],
 ['*', 'F', '2', '0', '0']]
Hidden state:
[['1', '1', '1', '0', '0'],
 ['1', 'M', '1', '1', '1'],
 ['3', '3', '2', '1', 'M'],
 ['M', 'M', '2', '1', '1'],
 ['M', 'M', '2', '0', '0']]
LLM Move:
<think>

</think>

row: 1, col: 1, action: reveal
Move correct!


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Board state:
[['*', 'F', '2', '0', '0'],
 ['2', 'F', '2', '0', '0'],
 ['1', '1', '2', '1', '1'],
 ['2', '2', '3', 'F', '3'],
 ['F', 'F', '*', '*', '*']]
Hidden state:
[['2', 'M', '2', '0', '0'],
 ['2', 'M', '2', '0', '0'],
 ['1', '1', '2', '1', '1'],
 ['2', '2', '3', 'M', '3'],
 ['M', 'M', '3', 'M', 'M']]
LLM Move:
<think>

</think>

row: 5, col: 3, action: reveal
Move correct!
Moves correct: 105/175
Moves incorrect: 70/175
